## Step 2 — create block-specific workspaces
**# of cells in notebook:** 1

**Purpose:** Create a separate working directory for every block selected for the block-splitting workflow. Each block directory contains a file geodatabase with the selected block feature and the buildings located within that block.

**Input:**

- `heterogeneous_largePop_blocks` — the block selection created in Step 1
- a citywide buildings layer
- `block_id` — the unique block identifier

**Output:**

- `heterogeneous_largePop_blocks` directory containing one folder for each selected block
- within each block folder, a block-specific file geodatabase containing:
  - `block` — the individual selected block feature
  - `buildings` — citywide buildings clipped to that block

**Main logic:**

**Cell 1 — Create the block-splitting workspace**

1. Iterates through the selected blocks and extracts the numeric portion of each `block_id`.
2. Creates one folder and one file geodatabase for each selected block.
3. Copies the individual block feature to the block-specific geodatabase as `block`.
4. Clips the citywide buildings layer to the block and writes the result as `buildings`.
5. Reports the number of processed, skipped, and failed blocks.


In [ ]:
import arcpy
import os
import re
import traceback

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

blocks_fc = r"E:\_johannesburg\_analysis\blocks\blocks.gdb\heterogeneous_largePop_blocks"

buildings_fc = r"E:\_johannesburg\_analysis\population\population.gdb\johannesburg_buildings_utm35s"

out_base_dir = r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"

block_id_field = "block_id"

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

os.makedirs(out_base_dir, exist_ok=True)

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

if not arcpy.Exists(blocks_fc):
    raise FileNotFoundError(f"Blocks layer does not exist:\n{blocks_fc}")

if not arcpy.Exists(buildings_fc):
    raise FileNotFoundError(f"Buildings layer does not exist:\n{buildings_fc}")

fields = [f.name for f in arcpy.ListFields(blocks_fc)]

if block_id_field not in fields:
    raise ValueError(f"Field '{block_id_field}' not found in:\n{blocks_fc}")

print("Inputs verified.")
print(f"Blocks:    {blocks_fc}")
print(f"Buildings: {buildings_fc}")
print(f"Output:    {out_base_dir}")

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def extract_actual_id(block_id_value):
    """
    Converts values like 'blk_397' to '397'.
    Returns None if the expected pattern is not found.
    """
    if block_id_value is None:
        return None

    text = str(block_id_value).strip()

    match = re.search(r"_(\d+)$", text)

    if match:
        return match.group(1)

    return None

# ------------------------------------------------------------
# Main loop
# ------------------------------------------------------------

processed = 0
skipped = 0
failed = 0

oid_field = arcpy.Describe(blocks_fc).OIDFieldName

with arcpy.da.SearchCursor(blocks_fc, [oid_field, block_id_field]) as cursor:
    for oid, block_id in cursor:

        actual_id = extract_actual_id(block_id)

        if actual_id is None:
            print(f"SKIPPING OBJECTID {oid}: could not extract numeric ID from block_id = {block_id}")
            skipped += 1
            continue

        folder_name = f"_{actual_id}"
        folder_path = os.path.join(out_base_dir, folder_name)

        gdb_name = f"{folder_name}.gdb"
        gdb_path = os.path.join(folder_path, gdb_name)

        out_block_fc = os.path.join(gdb_path, "block")
        out_buildings_fc = os.path.join(gdb_path, "buildings")

        print("\n------------------------------------------------------------")
        print(f"Processing OBJECTID {oid}")
        print(f"block_id: {block_id}")
        print(f"actual id: {actual_id}")
        print(f"Folder: {folder_path}")
        print(f"GDB:    {gdb_path}")

        try:
            # ------------------------------------------------
            # Create folder
            # ------------------------------------------------

            os.makedirs(folder_path, exist_ok=True)

            # ------------------------------------------------
            # Create geodatabase if needed
            # ------------------------------------------------

            if not arcpy.Exists(gdb_path):
                arcpy.management.CreateFileGDB(folder_path, gdb_name)
                print("Created geodatabase.")
            else:
                print("Geodatabase already exists.")

            # ------------------------------------------------
            # Make one-feature block layer
            # ------------------------------------------------

            where_clause = f"{arcpy.AddFieldDelimiters(blocks_fc, oid_field)} = {oid}"

            block_lyr = f"block_lyr_{actual_id}"

            if arcpy.Exists(block_lyr):
                arcpy.management.Delete(block_lyr)

            arcpy.management.MakeFeatureLayer(
                in_features=blocks_fc,
                out_layer=block_lyr,
                where_clause=where_clause
            )

            count_block = int(arcpy.management.GetCount(block_lyr)[0])

            if count_block != 1:
                raise RuntimeError(f"Expected 1 selected block, found {count_block}")

            # Copy selected block to block-specific GDB
            if arcpy.Exists(out_block_fc):
                arcpy.management.Delete(out_block_fc)

            arcpy.management.CopyFeatures(block_lyr, out_block_fc)
            print(f"Copied block feature to: {out_block_fc}")

            # ------------------------------------------------
            # Clip buildings by block polygon
            # ------------------------------------------------

            if arcpy.Exists(out_buildings_fc):
                arcpy.management.Delete(out_buildings_fc)

            arcpy.analysis.Clip(
                in_features=buildings_fc,
                clip_features=out_block_fc,
                out_feature_class=out_buildings_fc
            )

            building_count = int(arcpy.management.GetCount(out_buildings_fc)[0])
            print(f"Clipped buildings to: {out_buildings_fc}")
            print(f"Building features: {building_count}")

            # Clean up temporary layer
            arcpy.management.Delete(block_lyr)

            processed += 1

        except Exception as e:
            print(f"FAILED for OBJECTID {oid}, block_id {block_id}")
            print(str(e))
            print(traceback.format_exc())
            failed += 1

            try:
                if arcpy.Exists(block_lyr):
                    arcpy.management.Delete(block_lyr)
            except:
                pass

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n============================================================")
print("DONE")
print(f"Processed: {processed}")
print(f"Skipped:   {skipped}")
print(f"Failed:    {failed}")
print("============================================================")